In [ ]:
print('hi')

Okay so the final project ! 
and man this is complex or in the starting feels very so
dont know how oauth will work, how to connect to gmail/calendar -
how to do integrations 3rd party
how the state will flow and all
hopefully these gets answered as the project goes.
I will do it !

not clear or sure about the architecture as well
how many agents do we take?
one to clasify the intent.
one to actually schedule the time slot when we get info from calendar?
one to draft a reply?
or we shall not create many as it is unnessacary inc in no of agents?
what do we take in state?

In [2]:
#structured output for extraction
from typing import Optional
from typing import Annotated, TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import add_messages
from langchain_core.messages import SystemMessage
from dotenv import load_dotenv

class meetingDetails(TypedDict):
    sender_name : Optional[str] = 'No Name Provided'
    sender_email_address : str
    agenda : str
    requested_day : list[str]
    time_preference : str
    time_duration : int

In [3]:
#Structured output for general agent
from typing import Literal
class Intent(TypedDict) :
    intent : Literal['Meeting_Email', 'Not_Meeting_Email']

In [ ]:
#Structured Output for draft email
class draft_email_SO(TypedDict) : 
    subject : str
    body : str

In [ ]:
#defining the state

class State(TypedDict):
    messages : Annotated[list, add_messages]
    intent : str
    extracted_email_info : meetingDetails 
    needs_clarification : bool
    available_slots : dict
    draft_response : draft_email_SO
    recommended_slots : list
    draft_email_approved : bool
    thread_id : str
    

In [ ]:
#classifier LLM
load_dotenv(override=True)
general_llm = ChatOpenAI(model='gpt-5-nano')
general_llm_with_so = general_llm.with_structured_output(Intent)

In [ ]:
def classify_email_agent(state : State) :
    PROMPT = f''' 
    You are working as a agent for classifying the intent of e-mails provided to you or to the user. You have to 
    analyze the e-mail, understand the intent and categorize it accordingly.
    here's the email - 
    {state['messages']}
    - For emails that are proposing a interview/meeting time, label them as Meeting_Email
    - For emails whose intent is anything other than scheduling/proposing meetings/interview/dedicated time, label 
    them as Not_Meeting_Email 
    '''
    response = general_llm_with_so.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    print(response)
    return {
        'intent' : response['intent']
    }

In [ ]:
def post_classication_routing(state : State) :
    print('inside classification routing...')
    print(state['intent'])
    return 'handle_meeting_requests' if state['intent']=='Meeting_Email' else 'Not_Meeting_Email'

In [ ]:
def no_meeting_node(state : State) :
    return {
        'messages' : 'This is not a meeting request E-mail. So no action required from me.' 
    }

In [ ]:
extract_llm = ChatOpenAI(model='gpt-5-nano')
extract_llm_with_SO = extract_llm.with_structured_output(meetingDetails)

In [ ]:
def handle_meeting_requests(state : State) :
    PROMPT = f''' 
    You are an agent whose work is to extract important details from e-mails. 
    Here's the email - {state["messages"][-1].content}
    When you are provided with an e-mail,
    you have to extract things like :-
    - sender's name(if provided in email)
    - sender's email address
    - agenda - the agenda/motive for the meeting
    - requested day - the day they're proposing the meeting for
    - time-preference - their preferred/proposed time slot
    - time_duration - the duration of the meeting
    If the email contains non-specific information such as  :-
    "Any time next week"
    "Sometime after the holiday"
    "Whenever you're free"
    "June 25-27" 
    then make it as requires more clarification.
    '''

    response = extract_llm_with_SO.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    print(response)
    return {
        'extracted_email_info' : response
    }

In [ ]:
from collections import defaultdict
from datetime import datetime, timedelta
from calendar_service import list_upcoming_events

WORK_START_HOUR = 10
WORK_END_HOUR = 19
MEETING_DURATION = 30


def get_available_slots(state:State):
    print('inside get_available_slots ...')
    events = list_upcoming_events()
    events_by_day = defaultdict(list)
    for event in events:
        start = datetime.fromisoformat(event["start"]["dateTime"])
        end = datetime.fromisoformat(event["end"]["dateTime"])
        events_by_day[start.date()].append((start, end))
    
    print(f'refactored events_by_day - {events_by_day}') 
    available_slots = {}
    for day, busy_slots in events_by_day.items():
        busy_slots.sort(key=lambda x: x[0])
        tz = busy_slots[0][0].tzinfo
        
        work_start = datetime.combine(
            day,
            datetime.min.time()
        ).replace(
            hour=WORK_START_HOUR,
            tzinfo=tz
        )

        work_end = datetime.combine(
            day,
            datetime.min.time()
        ).replace(
            hour=WORK_END_HOUR,
            tzinfo=tz
        )

        cursor = work_start
        day_slots = []
        for event_start, event_end in busy_slots:
            while cursor + timedelta(minutes=MEETING_DURATION) <= event_start:
                day_slots.append(
                    (
                        cursor.strftime("%Y-%m-%d %H:%M"),
                        (cursor + timedelta(minutes=MEETING_DURATION)).strftime("%Y-%m-%d %H:%M")
                    )
                )
                cursor += timedelta(minutes=MEETING_DURATION)
            cursor = max(cursor, event_end)

        while cursor + timedelta(minutes=MEETING_DURATION) <= work_end:
            day_slots.append(
                (
                    cursor.strftime("%Y-%m-%d %H:%M"),
                    (cursor + timedelta(minutes=MEETING_DURATION)).strftime("%Y-%m-%d %H:%M")
                )
            )
            cursor += timedelta(minutes=MEETING_DURATION)
        available_slots[str(day)] = day_slots
    print(f'returning available slots - {available_slots}')
    
    return {
        'available_slots' : available_slots 
    }  

In [ ]:
from langchain_openai import ChatOpenAI 
draft_email_llm = ChatOpenAI(model='gpt-5-nano') 
draft_email_llm_with_SO = draft_email_llm.with_structured_output(draft_email_SO)

In [ ]:
def select_best_slots(available_slots, date, preferred_time):

    slots = available_slots.get(date, [])
    target_time = datetime.strptime(
        preferred_time,
        "%I:%M %p"
    )

    ranked = []
    for start, end in slots:
        start_dt = datetime.strptime(
            start,
            "%Y-%m-%d %H:%M"
        )
        diff = abs(
            (start_dt.hour * 60 + start_dt.minute)
            - (target_time.hour * 60 + target_time.minute)
        )
        ranked.append((diff, start, end))

    ranked.sort(key=lambda x: x[0])
    return ranked[:3]

In [ ]:
def select_best_slots_node(state: State):

    recommended_slots = select_best_slots(
        available_slots=state["available_slots"],
        date="2026-06-20",      # temporarily hardcoded 
        preferred_time=state["extracted_email_info"]["time_preference"]
    )
    print(f'recommended slots - {recommended_slots}')
    return {
        "recommended_slots": recommended_slots
    }

In [ ]:
def draft_reply(state : State) :
    print('inside draft_reply ...')
    PROMPT = f''' 
    You are working as a a personal e-mail drafting assistant for me. You are provided with a proposed meeting details
    and also, my available time slots. You have to draft a reply to the proposer, informing about the time slots which
    I will be available on, respond with the available time slots which are near the proposed meeting time. Respond with a maximum of 
    3-5 slots. 
    here arw the meetiong details : \n
    {state['extracted_email_info']} \n
    and here are my available slots \n
    {state['recommended_slots']}
    '''
    response = draft_email_llm_with_SO.invoke(
        input=[SystemMessage(content=PROMPT)]
    )
    print(response)
    return {
        'draft_response' : response
    }
    

In [ ]:
def approval_node(state: State) :
    '''implements HITL'''
    print()
    print(f'Here\'s the draft mail \n yes- {state["draft_response"]}\n')
    prof = input("Approve? Yes/No...") 
    print(prof)
    if prof.lower()=='yes':
        return {
            'draft_email_approved' : True
        }
    else:
        return {'draft_email_approved': False}

In [ ]:
from gmail_service import send_message

def send_email_node(state : State) :
    recipient = state['extracted_email_info']['sender_email_address']
    subject = state['draft_response']['subject']
    body = state['draft_response']['body'] 
    print('Sending email now ...')
    result = send_message(recipient, subject, body)
    print(result)
    return  {
        'thread_id' : result['threadId']
    }
    
    

In [ ]:
def email_not_approved_node(state : State) : 
    return {
        'messages' : ''' Draft email has not been approved ...... We need to work on this a bit more !'''
        }

In [ ]:
def approval_node_routing(state: State) :
    return 'send_email_action' if state['draft_email_approved'] else 'email_not_approved_action'

In [ ]:
import json
from pathlib import Path
import sqlite3, os 
from datetime import datetime

folder_path = Path(r'C:\AppyProjects\CustomGenAIProjects\ai_meeting_coordination_agent\email_data')

def store_email_data_db_node(state : State) :
    proposed_slots = json.dumps(state['recommended_slots'])
    conn = sqlite3.connect(folder_path/'email_data.db')
    cursor = conn.cursor()
    cursor.execute(
        (''' 
        INSERT INTO scheduling_conversations (
            threadId,    
            recipient_address, 
            proposed_slots,       
            status,         
            created_at,
            agenda              
        ) VALUES (?,?,?,?,?,?)
        '''
        ),
        (
            state['thread_id'], state['extracted_email_info']['sender_email_address'], proposed_slots,
            'Waiting_For_Reply', datetime.now().isoformat(), state['extracted_email_info']['agenda']
        )
    )
    conn.commit()
    #conn.close()
    print('Record saved successfully !')
    

In [ ]:
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State)

graph_builder.add_node('classify_email_agent', classify_email_agent)
graph_builder.add_node('no_meeting_node', no_meeting_node)
graph_builder.add_node('handle_meeting_requests', handle_meeting_requests)
graph_builder.add_node('get_available_slot',get_available_slots)
graph_builder.add_node('select_best_slots_node',select_best_slots_node)
graph_builder.add_node('draft_reply',draft_reply) 
graph_builder.add_node('approval_node',approval_node)
graph_builder.add_node('send_email_node',send_email_node)
graph_builder.add_node('email_not_approved_node',email_not_approved_node)
graph_builder.add_node('store_email_data_db_node',store_email_data_db_node)

graph_builder.add_conditional_edges('classify_email_agent', post_classication_routing, {'Not_Meeting_Email':'no_meeting_node', 'handle_meeting_requests':'handle_meeting_requests'})
graph_builder.add_edge(START, 'classify_email_agent')
graph_builder.add_edge('handle_meeting_requests', 'get_available_slot')
graph_builder.add_edge('get_available_slot', 'select_best_slots_node')
graph_builder.add_edge('select_best_slots_node', 'draft_reply')
graph_builder.add_edge('draft_reply', 'approval_node')
graph_builder.add_conditional_edges('approval_node', approval_node_routing, {'send_email_action' : 'send_email_node', 'email_not_approved_action' : 'email_not_approved_node'})
graph_builder.add_edge('send_email_node', 'store_email_data_db_node')

graph = graph_builder.compile()


In [ ]:
from IPython.display import display, Image
#display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import HumanMessage
con = ''' 
From: arpit.bharti8503@gmail.com
To: ron.manager@yourcompany.com
Date: Mon, June 15, 2026 at 10:00 AM
Subject: Request for Development Sync

Hi Team,
I hope you are having a productive week.
I would like to propose a brief sync meeting this coming Saturday, June 20, 2026, at 5:00 PM to review our latest development milestones and align on next steps.
Please let me know if this time works for you, or suggest an alternative slot if you have a conflict.

Best regards, 
Alex

''' 
state = {
    'messages' : [HumanMessage(content=con)] 
}
res = graph.invoke(state)

WORKFLOW PART 2

WHEN REPLY COMES THEN ....a new graph execution starts so...

In [4]:
#structured output for reply email data extration
class ReplyExtraction(TypedDict) : 
    selected_slot : Optional[str] 
    explanation : Optional[str] 
    intent : Literal['Accepted_slot', 'Rejected_slot', 'Request Reschedule','Needs Clarification']

In [5]:
class ConfirmationState(TypedDict) :
    reply_email : str 
    threadId : str 
    replyEmailExtractedDetails : ReplyExtraction
    conversationData:str 
    schedule_meeting : bool

In [6]:
reply_extraction_llm = ChatOpenAI(model='gpt-5-nano')
reply_extraction_llm_with_so = reply_extraction_llm.with_structured_output(ReplyExtraction)

In [7]:
def extractReplyEmailData (replyState : ConfirmationState) : 
    print('inside extractReplyEmailData...')
    
    SYSTEM= f''' You are provided an email and you have to extract important information from it so that the ceo can know 
    what is the other party's response to the original email was. 
    This is the received email - {replyState['reply_email']} 
    ''' 
    response = reply_extraction_llm_with_so.invoke( 
        input=[SystemMessage(content=SYSTEM)] 
    ) 
    print(response)
    return {
        'replyEmailExtractedDetails' : response 
    }

In [8]:
def dont_schedule_node(replyState : ConfirmationState) :
    return{
        'CHILLLLLLL' 
    }

In [ ]:
import sqlite3
from pathlib import Path
folder_path = Path(r'C:\AppyProjects\CustomGenAIProjects\ai_meeting_coordination_agent\email_data')

def fetch_conversation_from_db(replyState : ConfirmationState) : 
    print('inside fetch_conversation_from_db...')
    threadId = replyState['threadId']
    
    conn = sqlite3.connect(folder_path/'email_data.db')
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    data = cursor.execute( 
        (''' 
         SELECT * from scheduling_conversations where threadId = ? 
        '''
        ) , (threadId,) 
    )
    email_data = data.fetchone() 
    if email_data is not None:
        return {
            'conversationData' : email_data
        }
    else:
        return {
            'messages' : 'No email chain present for this e-mail'
            }

In [ ]:
def approve_meeting_node(replyState:ConfirmationState) :   
    print('inside approve_meeting_node')
    approve_meeting = f''' 
        {replyState['conversationData']['recipient_address']} has proposed a meeting for {replyState['conversationData']['agenda']} 
        We had given them these available slots - \n {replyState['conversationData']["proposed_slots"]} and he/she has chosen \n
        {replyState['replyEmailExtractedDetails']['selected_slot']}
        Are you okay with me to schedule a meeting at the confirmed time? 
        '''
    print(approve_meeting) 

    create_event = input('Schedule Meeting ?  Yes/No...') 
    if create_event.lower() == 'yes':
        return { 'schedule_meeting' : "Go_ahead" }
    else:
        return { 'schedule_meeting' : "dont_schedule" }

In [19]:
def schedule_routing(replyState : ConfirmationState) :
    return "create_event" if replyState["schedule_meeting"]=='Go_ahead' else 'dont_schedule'

In [20]:
#we need conditional routing here for the non-happy flow. maybe in V2

In [21]:
from calendar_service import create_event

def create_event_node(replyState : ConfirmationState) :
    print('inside create_event_node...')
    summary = replyState['conversationData']['agenda']
    location = 'Google Meet' 
    attendees = replyState['conversationData']['recipient_address']
    
    #selected_slot = "2026-06-20 17:00-17:30"
    selected_slot = replyState['replyEmailExtractedDetails']['selected_slot']
    date_part, time_range = selected_slot.split(" ")
    start_time_part, end_time_part = time_range.split('-')
    
    start_time = f'{date_part}{start_time_part}'
    end_time = f'{date_part}{end_time_part}'
    
    create_event(summary, location,start_time, end_time,attendees)
    
    

In [22]:
from langgraph.graph import StateGraph, START

graph_builder = StateGraph(ConfirmationState)

graph_builder.add_node('extractReplyEmailData', extractReplyEmailData)
graph_builder.add_node('fetch_conversation_from_db', fetch_conversation_from_db)
graph_builder.add_node('approve_meeting', approve_meeting_node)
graph_builder.add_node('create_event_node', create_event_node)
graph_builder.add_node('dont_schedule_node', dont_schedule_node)


graph_builder.add_edge(START, 'extractReplyEmailData')
graph_builder.add_edge('extractReplyEmailData', 'fetch_conversation_from_db')
graph_builder.add_edge('fetch_conversation_from_db', 'approve_meeting')
graph_builder.add_conditional_edges('approve_meeting', schedule_routing, {'create_event' : 'create_event_node', 'dont_schedule':'dont_schedule_node'})
graph_builder.add_edge('extractReplyEmailData', 'fetch_conversation_from_db')

confirmation_Graph = graph_builder.compile()


In [23]:
from IPython.display import Image, display
#display(Image(confirmation_Graph.get_graph().draw_mermaid_png()))


In [24]:
state = {
    "threadId": "19ef09cd4674f7a6",
    "reply_email": "5 PM works for me"
}

res = confirmation_Graph.invoke(state)
print(res['messages'][-1].content)

inside extractReplyEmailData...
{'selected_slot': '5 PM', 'explanation': 'The recipient confirmed that 5 PM works for them, indicating acceptance of the proposed time.', 'intent': 'Accepted_slot'}
inside fetch_conversation_from_db...
fetched email data from db is 
 - <sqlite3.Row object at 0x00000221E9A9E410>
inside approve_meeting_node
arpit.bharti8503@gmail.com


KeyError: 'selected_slot'

In [ ]:
# import sqlite3
# from pathlib import Path
# folder_path = Path(r'C:\AppyProjects\CustomGenAIProjects\ai_meeting_coordination_agent\email_data')
# conn = sqlite3.connect(folder_path/'email_data.db')
# cursor = conn.cursor()

# res = cursor.execute(
#     ''' 
#     select * from scheduling_conversations
#     '''
# )
# print(res.fetchall())